In [ ]:
# Check GPU

import os, torch
print("torch", torch.__version__, "| built for CUDA", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("LD_LIBRARY_PATH:", os.environ.get("LD_LIBRARY_PATH"))
try:
    torch.zeros(1).cuda()          # raises the real reason if CUDA cannot start
except Exception as e:
    print("ERROR:", e)

In [ ]:
# Imports

from pathlib import Path
import torch
import transformers
from tqdm import tqdm
import json, re, statistics
from collections import Counter
import datasets
from datasets import get_dataset_config_names, load_dataset

In [ ]:
# Constants
# Point this at the swissadmin_aligned corpus

PATH = "swissadmin_aligned_2014_05_26"

# Inspect Files
root = Path(PATH)
files = sorted(p for p in root.rglob("*") if p.is_file())
for p in files[:40]:
    print(f"{p.stat().st_size:>12,}  {p.relative_to(root)}")
print(len(files), "files in total")




In [ ]:
# Check Language distribution in the Corpus

def read_lines(path):
    with open(path, encoding="utf-8", errors="replace") as fh:
        return fh.read().split("\n")

folders = sorted(p for p in root.iterdir() if p.is_dir())

# Years per Folder
years = {}
for d in folders:
    a, b = d.name.split("-")
    ya = {f.name[:4] for f in d.glob(f"????{a}.txt")}
    yb = {f.name[:4] for f in d.glob(f"????{b}.txt")}
    years[d.name] = ya & yb
    print(d.name, "years with both sides:", min(years[d.name]), "-", max(years[d.name]),
          f"({len(years[d.name])} years)")

common = sorted(set.intersection(*years.values()))
print("\nyears present in all pairs:", common)

# Line counts per year
year = "2005"   # change if not in `common`
print()
for d in folders:
    a, b = d.name.split("-")
    la = read_lines(d / f"{year}{a}.txt")
    lb = read_lines(d / f"{year}{b}.txt")
    print(f"{d.name}: {a}={len(la)} lines, {b}={len(lb)} lines")

# is the English side the same in every pair that contains it?
en_de = read_lines(root / "de-en" / f"{year}en.txt")
en_fr = read_lines(root / "fr-en" / f"{year}en.txt")
en_it = read_lines(root / "it-en" / f"{year}en.txt")
print("\nen lines (de-en, fr-en, it-en):", len(en_de), len(en_fr), len(en_it))
print("identical de-en vs fr-en:", en_de == en_fr)
print("identical de-en vs it-en:", en_de == en_it)

In [ ]:
# Show samples of aligned sentences

de = read_lines(root / "de-en" / "2005de.txt")
en = read_lines(root / "de-en" / "2005en.txt")

for i in range(15):
    print(i, "DE:", repr(de[i][:150]))
    print(i, "EN:", repr(en[i][:150]))
    print()

In [ ]:
de_en = set(read_lines(root / "de-en" / "2005de.txt"))
de_fr = set(read_lines(root / "de-fr" / "2005de.txt"))
de_it = set(read_lines(root / "de-it" / "2005de.txt"))

print("distinct German lines: de-en", len(de_en), "| de-fr", len(de_fr), "| de-it", len(de_it))
print("de-en lines also in de-fr:", len(de_en & de_fr))
print("de-en lines also in de-it:", len(de_en & de_it))
print("in all three:", len(de_en & de_fr & de_it))

In [ ]:
# Check alignment and date template
# File specific constants

STRICT_DOC_START = False   # False: a dateline on EITHER side starts a new document
                           # True : only when BOTH sides have a dateline

JOBS = {"en_de": ("de-en", "en", "de"),
        "en_fr": ("fr-en", "en", "fr"),
        "en_it": ("it-en", "en", "it")}

root = Path(PATH)
assert root.is_dir(), f"{root.resolve()} does not exist"

# "Bern, 23.01.2013" / "Berna, ..." / "Berne, ..."
DATELINE = re.compile(r"^[A-ZÀ-Ý][\w\-' ]{1,30}, (\d{1,2})\.(\d{1,2})\.(\d{4})\b")

# Helper Functions

def read_lines(path):
    with open(path, encoding="utf-8", errors="replace") as fh:
        return fh.read().split("\n")

def clean(s):
    return re.sub(r"\s+", " ", s.replace("\xa0", " ")).strip()

def parse_date(s):
    """Return 'YYYY-MM-DD' if s starts with a valid dateline, else None."""
    m = DATELINE.match(s)
    if not m:
        return None
    d, mo, y = map(int, m.groups())
    if not (1 <= d <= 31 and 1 <= mo <= 12 and 1990 <= y <= 2030):
        return None
    return f"{y:04d}-{mo:02d}-{d:02d}"

# Extract Data

def extract_pairs(folder, l1, l2):
    d = root / folder
    years = sorted({f.name[:4] for f in d.glob(f"????{l1}.txt")} &
                   {f.name[:4] for f in d.glob(f"????{l2}.txt")})
    out, stats, issues = [], Counter(), []

    for year in years:
        A = read_lines(d / f"{year}{l1}.txt")
        B = read_lines(d / f"{year}{l2}.txt")
        assert len(A) == len(B), (folder, year, len(A), len(B))

        doc, sent, doc_date = 0, 0, None   # doc 0 = lines before the first dateline
        for line, (a, b) in enumerate(zip(A, B), 1):
            a, b = clean(a), clean(b)
            da, db = parse_date(a), parse_date(b)

            # --- classify the dateline situation
            if da and db:
                kind = "dateline_both_same_date" if da == db else "dateline_both_DIFFERENT_date"
            elif da or db:
                kind = "dateline_one_side_only"
            else:
                kind = None

            # --- new document?
            starts_doc = (da and db) if STRICT_DOC_START else (da or db)
            if starts_doc:
                doc, sent = doc + 1, 0
                doc_date = da or db
                stats[kind] += 1
                if not doc_date.startswith(year):
                    stats["doc_date_year_differs_from_file_year"] += 1
            elif kind:                         # dateline seen but not used as boundary (strict mode)
                stats[kind] += 1

            if kind in ("dateline_both_DIFFERENT_date", "dateline_one_side_only"):
                issues.append({"kind": kind, "id": f"{year}-line{line}", l1: a[:140], l2: b[:140]})

            # --- keep the pair
            if not a or not b:
                stats["dropped_empty_side"] += 1
                continue
            sent += 1
            out.append({"id": f"{year}-d{doc:04d}-s{sent:03d}", "year": year, "doc": doc,
                        "doc_date": doc_date, "sentence": sent, "line": line,
                        l1: a, l2: b})
    return out, stats, issues

# Run extraction over files

for name, (folder, l1, l2) in JOBS.items():
    pairs, stats, issues = extract_pairs(folder, l1, l2)
    with open(f"pairs_{name}.json", "w", encoding="utf-8") as f:
        json.dump(pairs, f, ensure_ascii=False, indent=1)

    doc_sizes = Counter((p["year"], p["doc"]) for p in pairs)
    sizes = sorted(doc_sizes.values())
    print(f"\n=== {name}: {len(pairs):,} pairs, {len(doc_sizes):,} documents ===")
    for k, v in sorted(stats.items()):
        print(f"  {k}: {v:,}")
    print(f"  sentences per document: min {sizes[0]}, median {statistics.median(sizes):.0f}, "
          f"mean {statistics.mean(sizes):.1f}, max {sizes[-1]}")

    # examples of suspicious datelines
    for kind in ("dateline_both_DIFFERENT_date", "dateline_one_side_only"):
        ex = [i for i in issues if i["kind"] == kind][:3]
        for i in ex:
            print(f"  [{kind}] {i['id']}")
            print(f"       {l1}: {i[l1]!r}")
            print(f"       {l2}: {i[l2]!r}")

    # the 3 biggest documents (possible missed datelines)
    for (year, doc), n in doc_sizes.most_common(3):
        first = next(p for p in pairs if p["year"] == year and p["doc"] == doc)
        print(f"  [largest doc] {year}-d{doc:04d}: {n} sentences, starts: {first[l1][:90]!r}")

In [ ]:
# Tokenise the extracted sentences on word-level

JOBS = {"en_de": ("en", "de"),
        "en_fr": ("en", "fr"),
        "en_it": ("en", "it")}

# a "word" = letters/digits, optionally joined by . , ' ’ - /
# anything else that is not whitespace = its own token (punctuation, symbols)

TOKEN = re.compile(r"\w+(?:[.,'’\-/]\w+)*|[^\w\s]|_", re.U)
HAS_WORD_CHAR = re.compile(r"[^\W_]", re.U)

PLACEHOLDER = 0     # score to be filled in later
PUNCT = -1          # punctuation and symbols

def tokenise(text):
    toks = TOKEN.findall(text)
    # sanity: nothing may be lost or invented
    assert "".join(toks) == re.sub(r"\s", "", text), text
    return [[t, PLACEHOLDER if HAS_WORD_CHAR.search(t) else PUNCT] for t in toks]

for name, (l1, l2) in JOBS.items():
    with open(f"pairs_{name}.json", encoding="utf-8") as f:
        pairs = json.load(f)

    out, stats = [], Counter()
    for p in pairs:
        rec = {k: v for k, v in p.items() if k not in (l1, l2)}   # keep all metadata
        for lang in (l1, l2):
            rec[lang] = tokenise(p[lang])        # [[token, label], ...]
            rec[f"{lang}_text"] = p[lang]        # original string, for traceability
            stats[f"tokens_{lang}"] += len(rec[lang])
            stats[f"punct_{lang}"] += sum(lab == PUNCT for _, lab in rec[lang])
        out.append(rec)

    with open(f"pairs_{name}_tokenised.json", "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False)       # compact: 50k records, indent would bloat it

    print(f"\n=== {name}: {len(out):,} pairs -> pairs_{name}_tokenised.json ===")
    for lang in (l1, l2):
        n, pn = stats[f"tokens_{lang}"], stats[f"punct_{lang}"]
        print(f"  {lang}: {n:,} tokens, {pn:,} punctuation ({pn/n:.1%}), "
              f"avg {n/len(out):.1f} tokens/sentence")
    ex = out[2]
    print("  example", ex["id"])
    print("   ", l1, ex[l1][:14], "...")
    print("   ", l2, ex[l2][:14], "...")

In [ ]:
# Check the configuration in the target dataset

SWISSGOV_RSD_DATASET = "ZurichNLP/SwissGov-RSD"

configs = get_dataset_config_names(SWISSGOV_RSD_DATASET)
print("configs:", configs)

for cfg in configs:
    ds = load_dataset(SWISSGOV_RSD_DATASET, cfg)
    print(f"\n=== config: {cfg} ===")
    print(ds)                                   # splits and row counts
    split = next(iter(ds.values()))
    print("features:", split.features)
    rec = split[0]
    for k, v in rec.items():
        print(f"  {k}: {str(v)[:150]!r}")

In [ ]:
# Check for overlaps with the target dataset

SWISSGOV_RSD_DATASET = "ZurichNLP/SwissGov-RSD"
SPLITS = ["test"]          # add "validation" here to check against it as well
JOBS = {"en_de": ("en", "de"), "en_fr": ("en", "fr"), "en_it": ("en", "it")}
N = 6            # n-gram size
THRESH = 0.5     # flag a pair if >= 50% of its n-grams occur in the test set

WORD = re.compile(r"\w+", re.U)
def norm_tokens(text):
    return WORD.findall(text.lower())

def ngrams(toks, n=N):
    return {tuple(toks[i:i + n]) for i in range(len(toks) - n + 1)}

def containment(text, ref):
    g = ngrams(norm_tokens(text))
    if not g:
        return None                       # fewer than N words: cannot be compared
    return sum(x in ref for x in g) / len(g)

# ---- reference n-grams from the chosen SwissGov-RSD split(s)
ds = load_dataset(SWISSGOV_RSD_DATASET)
cols = ["text_a_untokenized", "text_b_untokenized", "subset"]
rows = [r for s in SPLITS for r in ds[s].select_columns(cols)]
print(f"splits {SPLITS}: {len(rows)} rows, per subset:", dict(Counter(r["subset"] for r in rows)))

sg_en, sg_other = set(), {"de": set(), "fr": set(), "it": set()}
for r in rows:
    lang = r["subset"].split("_")[-1]
    assert lang in sg_other, r["subset"]
    sg_en |= ngrams(norm_tokens(r["text_a_untokenized"]))
    sg_other[lang] |= ngrams(norm_tokens(r["text_b_untokenized"]))
print(f"reference {N}-grams: en {len(sg_en):,}, " +
      ", ".join(f"{k} {len(v):,}" for k, v in sg_other.items()))

# compare every SwissAdmin pair

for name, (l1, l2) in JOBS.items():
    with open(f"pairs_{name}.json", encoding="utf-8") as f:
        pairs = json.load(f)

    flagged, short, bins = [], 0, Counter()
    for p in pairs:
        c1, c2 = containment(p[l1], sg_en), containment(p[l2], sg_other[l2])
        if c1 is None and c2 is None:
            short += 1
            continue
        best = max(c1 or 0, c2 or 0)
        bins["0" if best == 0 else "<0.2" if best < .2 else "<0.5" if best < .5
             else "<0.9" if best < .9 else ">=0.9"] += 1
        if best >= THRESH:
            flagged.append({"id": p["id"], "doc": (p["year"], p["doc"]),
                            l1: round(c1 or 0, 2), l2: round(c2 or 0, 2),
                            "text": p[l1][:110]})

    docs = Counter((p["year"], p["doc"]) for p in pairs)
    docs_hit = Counter(x["doc"] for x in flagged)
    docs_mostly = [d for d, k in docs_hit.items() if k / docs[d] > 0.5]

    print(f"\n=== {name}: {len(pairs):,} pairs ===")
    print(f"  too short for {N}-grams: {short:,}")
    print(f"  best containment bins: {dict(bins)}")
    print(f"  flagged (>= {THRESH}): {len(flagged):,} pairs "
          f"| docs with >=1 flagged: {len(docs_hit):,} | docs with >50% flagged: {len(docs_mostly):,}")
    for x in sorted(flagged, key=lambda x: -max(x[l1], x[l2]))[:6]:
        print(f"    {x['id']}  {l1}={x[l1]} {l2}={x[l2]}  {x['text']!r}")

    with open(f"overlap_{name}.json", "w", encoding="utf-8") as f:
        json.dump(flagged, f, ensure_ascii=False, indent=1)

In [ ]:

for name in ("en_de", "en_fr", "en_it"):
    pairs = json.load(open(f"pairs_{name}.json", encoding="utf-8"))
    nav = [p for p in pairs if "Navigation Menu" in p["en"]]
    en_counts = Counter(p["en"] for p in pairs)
    repeated = sum(1 for p in pairs if en_counts[p["en"]] >= 5 and len(p["en"]) > 20)
    print(f"{name}: {len(pairs):,} pairs | navigation line: {len(nav)} "
          f"| English line repeated >=5x: {repeated}")
    for t, n in en_counts.most_common(4):
        print(f"    {n:>4}x  {t[:90]!r}")

In [ ]:
# Clean the json files

JOBS = {"en_de": "de", "en_fr": "fr", "en_it": "it"}

STUBS = re.compile(
    r"Navigation Menu"
    r"|This document is available in German, French and Italian only",
    re.I)

for name, other in JOBS.items():
    with open(f"pairs_{name}_tokenised.json", encoding="utf-8") as f:
        pairs = json.load(f)
    leak_ids = {x["id"] for x in json.load(open(f"overlap_{name}.json", encoding="utf-8"))}

    kept, removed, reasons = [], [], Counter()
    for p in pairs:
        why = []
        if p["id"] in leak_ids:
            why.append("overlap_with_testset")
        if STUBS.search(p["en_text"]):
            why.append("english_stub_line")
        if why:
            reasons["+".join(why)] += 1
            removed.append({"id": p["id"], "reason": why, "en": p["en_text"][:120],
                            other: p[f"{other}_text"][:120]})
        else:
            kept.append(p)

    with open(f"pairs_{name}_tokenised_clean.json", "w", encoding="utf-8") as f:
        json.dump(kept, f, ensure_ascii=False)
    with open(f"removed_{name}.json", "w", encoding="utf-8") as f:
        json.dump(removed, f, ensure_ascii=False, indent=1)

    print(f"\n=== {name}: {len(pairs):,} -> {len(kept):,} kept, {len(removed):,} removed ===")
    for r, n in reasons.most_common():
        print(f"   {n:>5}  {r}")